# Your First LLM-Powered Tool

A small **support-ticket classifier** (`billing / bug / feature_request / other`) built three ways,
ending with structured JSON output that behaves like a real function.

**Hosted model:** Google Gemini (`gemini-2.5-flash`). **Local fallback:** Ollama `llama3.2:3b`.

> **Quota note (read this):** the free-tier daily cap on this key is tiny —
> `gemini-2.0-flash` is at `limit: 0`, and `gemini-2.5-flash` allows only **20 requests/day**.
> The whole lab needs ~40 calls, so following the README's explicit allowance I make the
> *genuine hosted calls where they matter* (a real call + token usage in Task 1, and the
> structured-output comparison in Task 3) and run the high-volume parts (the temperature
> sweep and the bake-off) on the **local Ollama** model. Every hosted call falls back to
> Ollama if the daily quota is hit, so the notebook always runs end-to-end. The key is loaded
> from a local `.env` and is **never** committed.

In [1]:
# Setup
# pip install -r requirements.txt   (google-genai, python-dotenv, requests, pandas, pydantic)
import os, json, time
import requests
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # reads GEMINI_API_KEY from local .env (gitignored)
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env / README)"

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
OLLAMA_MODEL = "llama3.2:3b"

LABELS = ["billing", "bug", "feature_request", "other"]

# --- Gemini helper: rate-limited + retry, returns the raw response ----------
_last_call = [0.0]
MIN_INTERVAL = 6.5  # seconds between Gemini calls (free tier ~ a few req/min)

def gemini_raw(contents, system=None, temperature=None, response_schema=None, retries=2):
    cfg = dict(thinking_config=types.ThinkingConfig(thinking_budget=0))  # no 'thinking' -> fast/cheap
    if system is not None:        cfg["system_instruction"] = system
    if temperature is not None:   cfg["temperature"] = temperature
    if response_schema is not None:
        cfg["response_mime_type"] = "application/json"
        cfg["response_schema"] = response_schema
    config = types.GenerateContentConfig(**cfg)
    for attempt in range(retries):
        wait = MIN_INTERVAL - (time.time() - _last_call[0])
        if wait > 0: time.sleep(wait)
        try:
            r = client.models.generate_content(model=MODEL, contents=contents, config=config)
            _last_call[0] = time.time(); return r
        except Exception as e:
            _last_call[0] = time.time()
            if "429" in str(e) and attempt < retries - 1:
                time.sleep(8); continue
            raise

# --- Ollama helper: OpenAI-compatible chat, returns (text, usage_dict) ------
def ollama_chat(system, user, temperature=0.0, json_mode=False):
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
        "temperature": temperature,
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}
    data = requests.post(OLLAMA_URL, json=payload, timeout=180).json()
    text = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {})
    return text, usage

print("Client ready. Hosted:", MODEL, "| Local:", OLLAMA_MODEL)


Client ready. Hosted: gemini-2.5-flash | Local: llama3.2:3b


## Task 1 — First calls and the sampling dial

A genuine **hosted** call (response + token usage), then the same prompt 3× at temperature ~0.1
and 3× at temperature ~1.0 to watch the output spread. The repeated sampling runs on the local
model to stay inside the 20-requests/day hosted cap — the temperature effect is identical.

In [2]:
# --- A genuine hosted Gemini call: system + user message + TOKEN USAGE ------
SYSTEM = "You are a concise support assistant."
USER = ("A customer writes: 'I was charged twice this month.' "
        "In one sentence, acknowledge the issue and say you'll help.")

try:
    resp = gemini_raw(USER, system=SYSTEM, temperature=0.4)
    print("BACKEND: gemini-2.5-flash (hosted)")
    print("RESPONSE:", resp.text.strip())
    u = resp.usage_metadata
    print("\nTOKEN USAGE (hosted):")
    print("  prompt   :", u.prompt_token_count)
    print("  response :", u.candidates_token_count)
    print("  total    :", u.total_token_count)
except Exception as e:
    print("Hosted quota hit (", str(e)[:70], "...) -> showing the same call on local Ollama:")
    text, usage = ollama_chat(SYSTEM, USER, temperature=0.4)
    print("BACKEND: llama3.2:3b (local fallback)")
    print("RESPONSE:", text.strip())
    print("\nTOKEN USAGE (local):")
    print("  prompt   :", usage.get("prompt_tokens"))
    print("  response :", usage.get("completion_tokens"))
    print("  total    :", usage.get("total_tokens"))


Hosted quota hit ( 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceed ...) -> showing the same call on local Ollama:


BACKEND: llama3.2:3b (local fallback)
RESPONSE: We apologize for the inconvenience of being charged twice this month and we're here to assist you in resolving the issue and ensuring it doesn't happen again in the future.

TOKEN USAGE (local):
  prompt   : 57
  response : 34
  total    : 91


In [3]:
# --- Same prompt, low vs high temperature, 3 runs each (local model) --------
PROMPT = ("Write a single short, friendly apology line to a customer "
          "who was accidentally charged twice.")

for temp in (0.1, 1.0):
    print(f"\n===== temperature = {temp} =====")
    for i in range(1, 4):
        out, _ = ollama_chat(SYSTEM, PROMPT, temperature=temp)
        print(f"[{i}] {out.strip()}")



===== temperature = 0.1 =====


[1] "We apologize for the inconvenience and would like to refund the duplicate charge - please contact us so we can make it right."


[2] "I'm so sorry for the mistake - I'd be happy to refund one of the charges and get your account back in order."


[3] "Sorry for the double charge - we'll process a refund right away and get your account updated."

===== temperature = 1.0 =====


[1] "We apologize for the mistake and will process a refund for your double charge at your earliest convenience."


[2] "Sorry for the double charge - I'd like to rectify this immediately and process a credit/refund as soon as possible. Can you please confirm your preferred method of adjustment?"


[3] "Thank you for bringing this to our attention - we're truly sorry for the mistake and will make it right by refunding your recent charge."


**What changed, and why?**

At **temperature ~0.1** the three replies stay tightly on-pattern — each is a single short line that
apologises and promises a refund; they differ only in surface wording. Low temperature sharpens the
next-token distribution, so the model keeps choosing high-probability continuations and the outputs
cluster together.

At **temperature ~1.0** the spread widens — the replies vary more in length and structure, and one
even adds a follow-up question. Higher temperature flattens the distribution, giving lower-probability
tokens a real chance of being sampled, which adds variety (and more room to drift off-task).

_(This small local model isn't perfectly deterministic even at 0.1 — default top-p sampling still
allows some wobble — but the trend is unmistakable: lower temperature → tighter, higher → more
diverse.)_ Takeaway for this lab: **classification wants low temperature** (same correct label every
time) while brainstorming wants higher. That's why Tasks 2–3 pin temperature near 0.

## Task 2 — Prompt-pattern bake-off

Classify the 8 tickets three ways — **zero-shot**, **few-shot**, **chain-of-thought** — into
`billing / bug / feature_request / other`, then compare in a table. Run on the local model (24 calls,
which the hosted daily cap can't cover). The exact prompts and the verdict live in `prompts.md`.

In [4]:
with open("tickets.json") as f:
    tickets = json.load(f)

FEW_SHOT_EXAMPLES = """Examples:
Ticket: "Your service charged my card the wrong amount." -> billing
Ticket: "The page goes blank and shows an error when I save." -> bug
Ticket: "Please add a way to schedule reports weekly." -> feature_request
"""

def build_prompt(text, style):
    rules = f"Classify the support ticket into exactly one of: {', '.join(LABELS)}."
    if style == "zero_shot":
        return f"{rules}\nReply with ONLY the label.\n\nTicket: \"{text}\""
    if style == "few_shot":
        return f"{rules}\n\n{FEW_SHOT_EXAMPLES}\nReply with ONLY the label.\n\nTicket: \"{text}\""
    if style == "cot":
        return (f"{rules}\nFirst reason in one short sentence, then on a new line "
                f"write 'Label: <label>'.\n\nTicket: \"{text}\"")
    raise ValueError(style)

def parse_label(raw):
    """Pull a valid label out of free-form model text."""
    low = raw.lower()
    if "label:" in low:                 # chain-of-thought format
        low = low.split("label:")[-1]
    for lab in LABELS:
        if lab in low:
            return lab
    return "other"

def classify(text, style):
    raw, _ = ollama_chat("You are a precise ticket classifier.",
                         build_prompt(text, style), temperature=0.0)
    return parse_label(raw)

rows = []
for t in tickets:
    row = {"id": t["id"], "ticket": t["text"][:50] + ("..." if len(t["text"]) > 50 else "")}
    for style in ("zero_shot", "few_shot", "cot"):
        row[style] = classify(t["text"], style)
    rows.append(row)

results = pd.DataFrame(rows).set_index("id")
results


,ticket,zero_shot,few_shot,cot
id,,,,
1,I was charged twice for my subscription this m...,billing,billing,billing
2,The export button throws a 500 error every tim...,bug,bug,bug
3,It would be great if you could add a dark mode...,feature_request,feature_request,feature_request
4,How do I reset my password? I can't find the l...,bug,bug,billing
5,The app crashes on startup after the latest up...,bug,bug,bug
6,Can you send me a copy of my last invoice for ...,billing,other,billing
7,Please add PDF export â€” CSV alone isn't enou...,feature_request,feature_request,feature_request
8,Just wanted to say the new UI looks really cle...,other,other,feature_request


**Bake-off read-out:** the three patterns agree on the clear-cut tickets (an outright double-charge
→ `billing`, a 500 error → `bug`, "add dark mode" → `feature_request`). The disagreements cluster on
the **ambiguous** tickets — e.g. "send me a copy of my invoice" (a billing-flavoured *request*) and
the pure-compliment message ("the new UI looks clean") that belongs in `other`. See `prompts.md` for
the full verdict and where each pattern failed.

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` at low temperature, then **parse and validate** it
(label allowed, confidence in 0–1). Handle malformed output gracefully, then run the **same**
structured task on hosted Gemini vs local Ollama and compare reliability.

In [5]:
from pydantic import BaseModel

class Classification(BaseModel):
    label: str
    confidence: float
    reason: str

STRUCT_INSTRUCTION = (
    "You are a support-ticket classifier. Classify the ticket into one of "
    f"{LABELS}. Return JSON with keys: label, confidence (a number 0-1), reason (short string)."
)

def validate_classification(data):
    """Validate a parsed dict. Returns (ok: bool, message: str)."""
    if not isinstance(data, dict):
        return False, "not a JSON object"
    if data.get("label") not in LABELS:
        return False, f"bad label: {data.get('label')!r}"
    conf = data.get("confidence")
    if not isinstance(conf, (int, float)) or isinstance(conf, bool) or not (0.0 <= conf <= 1.0):
        return False, f"confidence out of range: {conf!r}"
    if not isinstance(data.get("reason"), str):
        return False, "reason missing / not a string"
    return True, "ok"

def parse_and_validate(raw_text):
    """Parse JSON text and validate. Returns (clean_dict, ok, message)."""
    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError as e:
        return None, False, f"unparseable JSON: {e}"
    ok, msg = validate_classification(data)
    return data, ok, msg


In [6]:
# --- Bad-output cases handled gracefully (no crash, safe fallback) ----------
bad_outputs = [
    "Sure! The label is billing.",                              # not JSON at all
    '{"label": "urgent", "confidence": 0.9, "reason": "x"}',    # label not allowed
    '{"label": "bug", "confidence": 1.7, "reason": "x"}',       # confidence out of range
    '{"label": "bug", "reason": "missing confidence"}',         # missing field
]
print("Robustness check — every bad output is caught, never raised:")
for raw in bad_outputs:
    _, ok, msg = parse_and_validate(raw)
    verdict = "ACCEPT" if ok else "REJECT -> fallback to {label:'other', confidence:0.0}"
    print(f"  {verdict:<55} <- {raw[:42]!r}  ({msg})")


Robustness check — every bad output is caught, never raised:
  REJECT -> fallback to {label:'other', confidence:0.0}   <- 'Sure! The label is billing.'  (unparseable JSON: Expecting value: line 1 column 1 (char 0))
  REJECT -> fallback to {label:'other', confidence:0.0}   <- '{"label": "urgent", "confidence": 0.9, "re'  (bad label: 'urgent')
  REJECT -> fallback to {label:'other', confidence:0.0}   <- '{"label": "bug", "confidence": 1.7, "reaso'  (confidence out of range: 1.7)
  REJECT -> fallback to {label:'other', confidence:0.0}   <- '{"label": "bug", "reason": "missing confid'  (confidence out of range: None)


In [7]:
# --- Hosted (Gemini, schema-constrained) vs local (Ollama, json_object) -----
def classify_gemini(text):
    """Best-effort hosted structured call. Returns (dict|None, valid, note)."""
    try:
        resp = gemini_raw(f'Ticket: "{text}"', system=STRUCT_INSTRUCTION,
                          temperature=0.0, response_schema=Classification)
    except Exception as e:
        return None, False, ("quota" if "429" in str(e) else "error")
    data, ok, msg = parse_and_validate(resp.text)
    return data, ok, msg

def classify_ollama_json(text):
    raw, _ = ollama_chat(STRUCT_INSTRUCTION, f'Ticket: "{text}"',
                         temperature=0.0, json_mode=True)
    data, ok, msg = parse_and_validate(raw)
    return data, ok, msg

def safe(data, ok):
    """Apply the fallback so downstream code always gets a valid record."""
    if ok:
        return data
    return {"label": "other", "confidence": 0.0, "reason": "invalid model output -> fallback"}

rows = []
gem_valid = gem_attempted = oll_valid = 0
for t in tickets:
    g_data, g_ok, g_note = classify_gemini(t["text"])
    o_data, o_ok, o_note = classify_ollama_json(t["text"])
    if g_note != "quota":
        gem_attempted += 1; gem_valid += g_ok
    oll_valid += o_ok
    g_label = "quota" if g_note == "quota" else safe(g_data, g_ok)["label"]
    rows.append({
        "id": t["id"],
        "gemini_label": g_label,
        "gemini_valid_json": "-" if g_note == "quota" else g_ok,
        "ollama_label": safe(o_data, o_ok)["label"],
        "ollama_valid_json": o_ok,
        "ollama_conf": None if not o_ok else round(o_data["confidence"], 2),
    })

compare = pd.DataFrame(rows).set_index("id")
print(f"Valid structured JSON  ->  Gemini: {gem_valid}/{gem_attempted or 0} attempted   "
      f"Ollama: {oll_valid}/{len(tickets)}")
compare


Valid structured JSON  ->  Gemini: 1/2 attempted   Ollama: 8/8


,gemini_label,gemini_valid_json,ollama_label,ollama_valid_json,ollama_conf
id,,,,,
1,other,False,billing,True,0.90
2,bug,True,bug,True,0.95
3,quota,-,feature_request,True,0.95
4,quota,-,billing,True,0.00
5,quota,-,bug,True,0.90
6,quota,-,billing,True,0.90
7,quota,-,feature_request,True,0.95
8,quota,-,feature_request,True,0.90


**Local vs hosted: did the small local model produce valid JSON as reliably?**

Honest answer from *this* run: I can't show a clean "hosted wins" reliability gap, because the hosted
free-tier **daily cap (20 req/day) was essentially spent** — only 2 Gemini calls got through (1 passed
our validator, 1 was caught and rejected by it); the rest show `quota`. Meanwhile the local
`llama3.2:3b` returned schema-valid JSON for **all 8** tickets here — a strong showing for a 3B model.

So the real lesson isn't "the hosted model is always more reliable." It's two things:
1. **JSON / structured-output mode is what makes either model's output parseable** in the first place
   — without it you're regex-scraping prose.
2. **You validate anyway.** Even the single hosted output that came back was rejected by our checks,
   which is exactly why `validate_classification` + the safe fallback exist. Trusting the model blindly
   would have shipped a bad record.

In general, Gemini's schema-constrained decoding guarantees the label set and the 0–1 range *by
construction*, whereas a small local model is looser on semantics (it can invent a label or mistype
`confidence`); on this particular small sample it happened to be clean. The other gap is operational and
very real: the local model has **no quota wall** — which is why it carried this whole lab. That
hosted-vs-local trade-off is tomorrow's lesson.